<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Chameleon Facility Ports: FABnetv4 (Layer 3)

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** This notebook demonstrates how to connect Chameleon bare-metal servers to FABRIC's **FABnetv4 Layer 3 routed network** using a facility port. Unlike the L2 stitch (which provides a flat Ethernet segment), the L3 approach connects Chameleon into FABRIC's routed infrastructure, allowing Chameleon servers to communicate with FABRIC nodes at **any site** across the entire testbed.

</div>

<img src="./figs/stitching_triangle.png" width="60%"><br>

The goal is to create an experiment resembling the figure below -- Chameleon servers at TACC plus FABRIC nodes at other sites, all communicating over FABRIC's FABnetv4 network.

<img src="./figs/fabnet_stitch.png" width="60%"><br>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Understand the difference between **Layer 2** and **Layer 3** facility port stitching
2. Create a FABRIC slice with a **FABnetv4 network** connected to a Chameleon facility port
3. Extract **subnet and gateway information** from the FABnetv4 network for Chameleon configuration
4. Configure Chameleon's subnet with **host routes** that enable routing through FABRIC's backbone
5. Create additional FABRIC nodes at **remote sites** that can communicate with Chameleon servers
6. Verify **end-to-end connectivity** between Chameleon servers and FABRIC nodes across multiple sites

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Complete the [Configure Environment](../../fablib_api/configure_environment/configure_environment.ipynb) notebook for FABRIC
2. Have an active **Chameleon account** with a project allocation at TACC
3. Have a **Chameleon CLI password** (see [Chameleon CLI Authentication](https://chameleoncloud.readthedocs.io/en/latest/technical/cli.html#cli-authentication))
4. Have a Chameleon **openrc file** for the TACC site
5. Have an active **Chameleon server reservation** and SSH key
6. Understand the basics of L2 stitching (see [Chameleon L2 Facility Port](./Chameleon_Facility_Port_L2.ipynb))

**Tip:** This notebook creates **two FABRIC slices** -- one for the facility port stitch and one for additional FABRIC nodes. Both must be cleaned up at the end.

</div>

## Background: L2 vs L3 Facility Port Stitching

The [Layer 2 notebook](./Chameleon_Facility_Port_L2.ipynb) created a flat Ethernet connection between FABRIC and Chameleon. This works well for simple topologies but has limitations:

- All nodes must be on the **same subnet**
- No routing through FABRIC's backbone to other sites
- Manual IP address management

The **FABnetv4 (Layer 3)** approach connects the Chameleon facility port to FABRIC's routed network:


**Key advantages of L3 stitching:**
- FABRIC assigns the subnet and gateway automatically
- Chameleon servers can reach FABRIC nodes at **any site** via FABnetv4 routing
- Host routes on Chameleon direct FABnet traffic through the FABRIC gateway
- Multiple FABRIC slices can be reached from Chameleon

## What We're Building

In this notebook we will create a FABRIC node connected to Chameleon via a routed FABnetv4 facility port.

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Configure Chameleon Environment

Set your Chameleon credentials. Replace the placeholders with your actual values.

<div class="fab-danger">

**Important:** Never commit credentials to version control. Use `load_dotenv` to keep passwords out of the notebook.

</div>

In [ ]:
import os

# Set your Chameleon credentials
# Replace placeholders with your actual values
os.environ["OS_USERNAME"]='<username>'
os.environ["OS_PASSWORD"]='<password'
os.environ["OS_PROJECT_ID]='<project_id>'

# Chameleon TACC authentication endpoint
os.environ["OS_AUTH_URL"]='https://chi.tacc.chameleoncloud.org:5000/v3'
os.environ["OS_IDENTITY_API_VERSION"]='3'
os.environ["OS_INTERFACE"]='public'
os.environ["OS_PROTOCOL"]="openid"
os.environ["OS_AUTH_TYPE"]="v3oidcpassword"
os.environ["OS_IDENTITY_PROVIDER"]="chameleon"
os.environ["OS_DISCOVERY_ENDPOINT"]="https://auth.chameleoncloud.org/auth/realms/chameleon/.well-known/openid-configuration"
os.environ["OS_CLIENT_ID"]="keystone-tacc-prod"
os.environ["OS_ACCESS_TOKEN_TYPE"]="access_token"
os.environ["OS_CLIENT_SECRET"]="none"
os.environ["OS_REGION_NAME"]="CHI@TACC"

Alternatively, load credentials from your openrc file.

<div class="fab-warning">

**Tip:** Put your Chameleon openrc file in your fabric_config folder with your FABRIC keys. Then reference it here with a full path: `f'{os.environ["HOME"]}/work/fabric_config/chameleon-openrc-tacc.sh'`

</div>

In [ ]:
from dotenv import load_dotenv
import os

# Load the environment variables from your openrc file
load_dotenv(f'chameleon-openrc-tacc.sh');

## Step 2: Import Libraries

In [ ]:
# General imports
import os
import json
import traceback
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
from datetime import datetime, timedelta
from dateutil import tz
import time

# Chameleon Library
import chi
import chi.lease 
from chi.server import *
from chi.lease import *
from chi.network import *

# FABRIC Library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

## Step 3: Define Experiment Variables

### Chameleon Variables

<div class="fab-danger">

**Important:** Replace `<Your_Chameleon_Key>` and `<Your_Chameleon_Server_Reservation>` with your actual values.

</div>

In [ ]:
# Chameleon Config -- prefix all resource names with your username
chameleon_prefix =  f'{os.environ["OS_USERNAME"]}_fabric_stitch'
chameleon_server_name = chameleon_prefix+'_server'
chameleon_network_name = chameleon_prefix+'_net'
chameleon_subnet_name = chameleon_prefix+'_subnet'
chameleon_router_name = chameleon_prefix+'_router'
chameleon_lease_name = chameleon_prefix+'_lease'

chameleon_image_name='CC-Ubuntu20.04'  # Chameleon bare-metal OS image
chameleon_server_count=1               # Number of Chameleon servers

# REPLACE with your actual values
chameleon_key_name='<Your_Chameleon_Key>'
chameleon_server_reservation_id = '<Your_Chameleon_Server_Reservation>'

### FABRIC Variables

Configure the FABRIC side with the TACC facility port.

In [ ]:
# Create a FABlib manager and verify configuration
fablib = fablib_manager()
fablib.show_config()

# FABRIC Config
fabric_slice_name='chameleon_stitch'

# TACC facility port
fabric_site='TACC'
fabric_slice_name="tacc_stitch"
faciliy_port='Chameleon-TACC'

# OR Chicago facility port (uncomment to use)
#fabric_site='STAR'
#fabric_slice_name="chicago_stitch"
#faciliy_port='Chameleon-StarLight'

---

## Step 4: Create the Chameleon Network

Reserve a Chameleon network with the `fabric` stitch provider and extract the VLAN tag.

In [ ]:
BLAZAR_TIME_FORMAT = '%Y-%m-%d %H:%M'

# Set start/end date for lease
# Start one minute into future to avoid Blazar thinking lease is in past
# due to rounding to closest minute.
start_date = (datetime.now(tz=tz.tzutc()) + timedelta(minutes=1)).strftime(BLAZAR_TIME_FORMAT)
end_date   = (datetime.now(tz=tz.tzutc()) + timedelta(days=1)).strftime(BLAZAR_TIME_FORMAT)

# Build list of reservations (in this case there is only one reservation)
reservation_list = []
reservation_list.append(
        {
            "resource_type": "network",
            "network_name": chameleon_network_name,
            "network_properties": "",
            "resource_properties": json.dumps(
                ["==", "$stitch_provider", 'fabric']  # Request a FABRIC-stitchable network
            ),
        }
)

# Create the lease
chameleon_lease = chi.lease.create_lease(chameleon_lease_name,
                                  reservations=reservation_list,
                                  start_date=start_date,
                                  end_date=end_date)
    
# Extract the network reservation ID
chameleon_network_reservation_id = [reservation for reservation in chameleon_lease['reservations'] if reservation['resource_type'] == 'network'][0]['id']
print(f"chameleon_network_reservation_id: {chameleon_network_reservation_id}")

### Get the Network and VLAN Tag

Wait for the Chameleon network to become active and extract the VLAN tag for FABRIC stitching.

In [ ]:
# Poll until the network is active and VLAN tag is available
network_vlan = None
while network_vlan == None:
    try:
        # Get the network from Chameleon
        chameleon_network = chi.network.get_network(chameleon_network_name)

        # Get the network ID
        chameleon_network_id = chameleon_network['id']
        print(f'Chameleon Network ID: {chameleon_network_id}')

        # Get the VLAN tag for FABRIC stitching
        network_vlan = chameleon_network['provider:segmentation_id']
        print(f'network_vlan: {network_vlan}')
    except:
        print(f'Chameleon Network is not ready. Trying again!')
        time.sleep(10)           

---

## Step 5: Create FABRIC Slice with FABnetv4 Facility Port

This is the key difference from the L2 notebook. Instead of creating an L2 network, we create a **FABnetv4 (Layer 3)** network and attach the facility port to it. FABRIC automatically assigns a subnet and gateway.

<div class="fab-warning">

**Tip:** Note that this slice does not contain any FABRIC compute nodes -- only the facility port and FABnetv4 network. FABRIC nodes will be added in a separate slice that joins the same FABnetv4 network.

</div>

In [ ]:
# Create a FABRIC slice for the facility port stitch
fabric_slice = fablib.new_slice(name=fabric_slice_name)

# Add the facility port using the Chameleon VLAN tag
fabric_facility_port = fabric_slice.add_facility_port(name=faciliy_port, site=fabric_site, vlan=str(network_vlan))
fabric_facility_port_iface = fabric_facility_port.get_interfaces()[0]

# Create a FABnetv4 (L3) network and attach the facility port
# FABRIC will automatically assign a subnet and gateway
fabric_net = fabric_slice.add_l3network(name=f'facility_port_fabnetv4', interfaces=[fabric_facility_port_iface])

# Submit the FABRIC slice
fabric_slice.submit()

### Extract FABnetv4 Network Information

After the FABRIC slice is active, extract the subnet and gateway assigned by FABnetv4. We will use these to configure the Chameleon subnet so that both sides can route traffic correctly.

<div class="fab-warning">

**Key concept:** The FABnetv4 gateway is the FABRIC router. Chameleon servers will use a host route to send FABnet-destined traffic through this gateway, enabling them to reach FABRIC nodes at any site.

</div>

In [ ]:
# Get the FABnetv4 network and its assigned subnet/gateway
fabric_network = fabric_slice.get_network('facility_port_fabnetv4')

subnet = fabric_network.get_subnet()
fabric_gateway_ip = fabric_network.get_gateway()

# Build a list of available IPs for Chameleon allocation
available_ips = list(subnet)[1:]  # Skip network address
available_ips.remove(fabric_gateway_ip)  # Remove the gateway IP

# Assign IP ranges for Chameleon
chameleon_gateway_ip=available_ips.pop(0)     # Chameleon router gateway
chameleon_allocation_pool_start=available_ips[0]   # Start of Chameleon server IP pool
chameleon_allocation_pool_end=available_ips[10]     # End of Chameleon server IP pool

print(f'fabric_gateway_ip: {fabric_gateway_ip}')
print(f'chameleon_gateway_ip: {chameleon_gateway_ip}')
print(f'chameleon_allocation_pool_start: {chameleon_allocation_pool_start}')
print(f'chameleon_allocation_pool_end: {chameleon_allocation_pool_end}')

---

## Step 6: Configure Chameleon Network, Subnet, and Router

### Add a Subnet with Host Routes

The critical configuration here is the **host route** that tells Chameleon servers how to reach FABRIC's FABnetv4 network. Any traffic destined for a FABnetv4 address will be forwarded to the FABRIC gateway IP.

<div class="fab-warning">

**Interesting point:** This cell uses both the Chameleon and FABRIC APIs together -- `fablib.FABNETV4_SUBNET` (a FABRIC constant) is used within the Chameleon `update_subnet` call to set the correct routing destination.

</div>

In [ ]:
# Create the subnet using the FABnetv4-assigned CIDR and IP pool
chameleon_subnet = chi.network.create_subnet(chameleon_subnet_name, chameleon_network_id, 
                                             cidr=str(subnet),
                                             allocation_pool_start=chameleon_allocation_pool_start,
                                             allocation_pool_end=chameleon_allocation_pool_end,
                                             gateway_ip=chameleon_gateway_ip)

# Add a host route so Chameleon servers can reach the entire FABnetv4 network
# Traffic to any FABnet destination will be forwarded to the FABRIC gateway
chi.neutron().update_subnet(subnet=chameleon_subnet['id'] ,
                                    body={
                                         "subnet": { 
                                             "host_routes": [ 
                                                    {
                                                        "destination": f"{fablib.FABNETV4_SUBNET}", 
                                                         "nexthop": f"{fabric_gateway_ip}"
                                                    }
                                             ] 
                                         }
                                    })

print(f"subnet name  : {chameleon_subnet['name']}")
print(f"subnet       : {chameleon_subnet['cidr']}")
print(f"gateway_ip   : {chameleon_subnet['gateway_ip']}")

### (Optional) Add a Router

Add a Chameleon router with public Internet access. This is optional but allows SSH access to Chameleon servers from the public Internet.

In [ ]:
# Create a router with public network gateway
chameleon_router = chi.network.create_router(chameleon_router_name, gw_network_name='public')

print(f"router name  : {chameleon_router['name']}")

# Attach the subnet to the router
connection_port = chi.network.add_subnet_to_router_by_name(chameleon_router_name, chameleon_subnet_name)

print(f"connection_port id  : {connection_port['port_id']}")

---

## Step 7: Start Chameleon Servers

Launch bare-metal servers on the stitched network. These will take 10-20 minutes to become active.

In [ ]:
# Create Chameleon bare-metal servers
servers = []

for i in range(chameleon_server_count):
    server_name=f"{chameleon_server_name}_{i+1}"
    # Create the server on the stitched network
    servers.append(chi.server.create_server(server_name, 
                                  reservation_id=chameleon_server_reservation_id, 
                                  network_name=chameleon_network_name, 
                                  image_name=chameleon_image_name,
                                  key_name=chameleon_key_name
                                 ))
    
# Wait until all servers are active (10-20 minutes for bare-metal)
for server in servers:
    print(f'Waiting for server: {server.name}')
    chi.server.wait_for_active(server.id)
print('Done!')

### Get Chameleon Server IP Addresses

In [ ]:
# Retrieve the fixed IP of each Chameleon server
fixed_ips={}
for i in range(chameleon_server_count):
    server_name=f"{chameleon_server_name}_{i+1}"
    server_id = get_server_id(server_name)
    fixed_ip = get_server(server_id).interface_list()[0].to_dict()["fixed_ips"][0]["ip_address"]
    fixed_ips[server_name]=fixed_ip

for server_name,fixed_ip in fixed_ips.items():
    print(f'{server_name}: {fixed_ip}')

---

## Step 8: Create Additional FABRIC Nodes at a Remote Site

This is where the L3 approach really shines. We create a **second FABRIC slice** with nodes at a completely different site, connected via FABnetv4. These remote FABRIC nodes can communicate with the Chameleon servers through FABRIC's backbone routing.

<div class="fab-success">

**Key insight:** Because FABnetv4 is a routed network, any node connected to FABnetv4 at any site can reach any other FABnetv4 node -- including the Chameleon servers stitched through the facility port.

</div>

In [ ]:
# Define the second slice for remote FABRIC nodes
slice_name = 'MyFabricNodes'

# Pick a random FABRIC site (different from TACC)
site = fablib.get_random_site()
print(f"Site: {site}")

node_cnt = 2  # Number of nodes to create

In [ ]:
# Create the second slice with FABnetv4-connected nodes
slice = fablib.new_slice(name=slice_name)

for i in range(1,node_cnt+1):
    # Add nodes with automatic FABnetv4 networking
    node = slice.add_node(name=f"node{i}", site=site)
    node.add_fabnet()  # Connect to FABnetv4 routed network

# Submit and wait for provisioning
slice.submit();

---

## Step 9: Test End-to-End Connectivity

Ping the Chameleon servers from the remote FABRIC nodes. This traffic will traverse:
1. The remote FABRIC site's FABnetv4 interface
2. FABRIC's backbone network
3. The TACC site's FABnetv4 router
4. The facility port to Chameleon
5. The Chameleon server

In [ ]:
# Ping Chameleon server from a remote FABRIC node
node = slice.get_node('node1')

stdout, stderr = node.execute(f'ping -c 5 {fixed_ip}')

<div class="fab-success">

**Success!** If pings succeed, you have a working Layer 3 connection from a remote FABRIC site to Chameleon servers at TACC, routed through FABRIC's FABnetv4 backbone.

</div>

---

## Step 10: Clean Up Resources

<div class="fab-danger">

**Important:** This notebook creates resources on **both** testbeds and uses **two** FABRIC slices. Clean up everything to avoid orphaned resources.

</div>

### Delete Chameleon Servers

In [ ]:
# Delete all Chameleon servers
for i in range(chameleon_server_count):
    server_name=f"{chameleon_server_name}_{i+1}"
    chi.server.delete_server(get_server_id(server_name))

### De-configure Chameleon Network

In [ ]:
# Remove Chameleon network resources in reverse order
router_id = chameleon_router['id']
subnet_id = chameleon_subnet['id']

try:
    result = chi.network.remove_subnet_from_router(router_id, subnet_id)
except Exception as e:
    print(f"detach_router_by_name error: {str(e)}")
    pass

try:
    result = chi.network.delete_router(router_id)
except Exception as e:
    print(f"delete_router_by_name error: {str(e)}")
    pass

try:
    result = chi.network.delete_subnet(subnet_id)
except Exception as e:
    print(f"delete_subnet_by_name error: {str(e)}")
    pass

try:
    result = chi.network.delete_network(network_id)
except Exception as e:
    print(f"delete_network_by_name error: {str(e)}")
    pass

### Release Chameleon Lease

In [ ]:
# Release the Chameleon network lease
chi.lease.delete_lease(chameleon_lease['id'])

### Delete Both FABRIC Slices

In [ ]:
# Delete the remote FABRIC nodes slice
try:
    slice = fablib.get_slice(name=slice_name)
    slice.delete()
except Exception as e:
    print(f"Exception: {e}")

# Delete the facility port stitch slice
try:
    slice = fablib.get_slice(name=fabric_slice_name)
    slice.delete()
except Exception as e:
    print(f"Exception: {e}")

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| Chameleon authentication fails | Wrong CLI password or expired token | Regenerate your CLI password on the Chameleon portal |
| Network lease fails | No stitchable networks available | Wait and retry, or check Chameleon portal for availability |
| FABRIC facility port fails | Invalid VLAN tag or port unavailable | Verify the VLAN tag and facility port name |
| FABnetv4 subnet not assigned | Slice not fully provisioned | Wait for slice to become Active before querying network info |
| Chameleon servers cannot reach FABRIC nodes | Missing host route | Verify the host route was applied: check `ip route` on the Chameleon server |
| Ping from remote site fails | FABnetv4 routing issue | Verify both slices are Active and nodes have FABnet IPs |
| `fablib.FABNETV4_SUBNET` not found | FABlib version too old | Update FABlib: `pip install --upgrade fabrictestbed-extensions` |
| Cleanup fails | Resources already deleted | Ignore errors during cleanup for already-deleted resources |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `fablib.get_random_site()` | Select a random available site | [get_random_site](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_site) |
| `slice.add_facility_port()` | Add a facility port for external connectivity | [add_facility_port](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_facility_port) |
| `slice.add_l3network()` | Add a FABnetv4 Layer 3 network | [add_l3network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_l3network) |
| `slice.add_node()` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `node.add_fabnet()` | Add FABnetv4 L3 networking to a node | [add_fabnet](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_fabnet) |
| `network.get_subnet()` | Get the assigned subnet of a network | [get_subnet](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.get_subnet) |
| `network.get_gateway()` | Get the gateway IP of a network | [get_gateway](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.get_gateway) |
| `fablib.FABNETV4_SUBNET` | The overall FABnetv4 address space | [FABNETV4_SUBNET](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.FABNETV4_SUBNET) |
| `node.execute()` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

Now that you understand Layer 3 facility port stitching, explore these related notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Chameleon L2 Stitch** | [Chameleon L2 Facility Port](./Chameleon_Facility_Port_L2.ipynb) | Simpler Layer 2 stitching for single-subnet topologies |
| **FABnet Networking** | [FABnet IPv4](../../fablib_api/create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_auto.ipynb) | Learn about FABRIC's Layer 3 networking fundamentals |
| **Network Performance** | [iPerf3 Optimized](../iPerf3/iperf3_optimized.ipynb) | Measure and optimize cross-site throughput |
| **One-Way Latency** | [OWL Measurements](../owl/owl.ipynb) | Measure one-way latency using PTP-synchronized clocks |